# Mamba3Yolo — Real YOLO Training + mAP (Option B + C)

Clean re-train with **stronger loss** (DFL + multi-positive assignment + CIoU + BCE) and periodic **mAP@0.5**.

### What you get
- Real detection loss from epoch 1
- Multi-positive grid assignment (center + 3×3)
- Distribution Focal Loss (DFL)
- mAP@0.5 every 10 epochs
- Paper-ready curves + best.pt + summary JSON

### How to use
1. Add brain-tumor dataset (already present)  
2. **Run All** (first cell installs / restarts if needed)  
3. Wait ~2–4 h on T4 for 100 epochs (or reduce `EPOCHS`)

## 1. Install + Setup

In [ ]:
import subprocess, sys, os
def pip(*a):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir"] + list(a))
print("Installing deps...")
pip("einops", "thop", "seaborn", "timm", "opencv-python-headless", "torchmetrics", "pycocotools")
try:
    pip("causal-conv1d>=1.4.0", "mamba-ssm", "--no-build-isolation")
    print("mamba-ssm OK")
except Exception:
    print("mamba-ssm fallback will be used")
print("Ready. If this is the first run of the session, restart kernel once then Run All again.")

## 2. Imports + model + data

In [ ]:
import os, sys, gc, json, math, random, time, warnings
from pathlib import Path
from datetime import datetime
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torchvision.transforms as T

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE=="cuda" else "cpu")

WORK = Path("/kaggle/working")
PROJ = WORK / "Mamba3Yolo"
if not PROJ.exists():
    import subprocess
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/ShMazumder/Mamba3Yolo.git", str(PROJ)])
sys.path.insert(0, str(PROJ))
os.chdir(PROJ)
print("Project:", PROJ)

from src.models.mamba3yolo import build_mamba3yolo
from src.blocks.mamba3_odss import HAS_MAMBA3
print("HAS_MAMBA3:", HAS_MAMBA3)

# Data
DATA_ROOT = Path("/kaggle/input/datasets/organizations/ultralytics/brain-tumor")
IMG_DIR = DATA_ROOT / "brain-tumor" / "valid" / "images"
LBL_DIR = DATA_ROOT / "brain-tumor" / "valid" / "labels"
if not IMG_DIR.exists():
    for p in DATA_ROOT.rglob("*.jpg"):
        IMG_DIR = p.parent
        LBL_DIR = IMG_DIR.parent / "labels"
        break
assert IMG_DIR.exists(), f"No images under {DATA_ROOT}"
print("Images:", IMG_DIR, "| Labels:", LBL_DIR)

NC = 9
IMGSZ = 640
EPOCHS = 100          # paper setting; set 30 for quick test
BATCH = 4
LR = 1e-3
print(f"nc={NC} imgsz={IMGSZ} epochs={EPOCHS} batch={BATCH} lr={LR}")

## 3. Dataset

In [ ]:
class YoloDS(Dataset):
    def __init__(self, img_dir, lbl_dir, imgsz=640, max_n=None, augment=False):
        self.imgsz = imgsz
        self.augment = augment
        self.paths = sorted(list(Path(img_dir).glob("*.jpg")) + list(Path(img_dir).glob("*.png")))
        if max_n: self.paths = self.paths[:max_n]
        self.lbl_dir = Path(lbl_dir)
        self.tf = T.Compose([T.Resize((imgsz, imgsz)), T.ToTensor()])
        print(f"YoloDS: {len(self.paths)} images (augment={augment})")
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        p = self.paths[i]
        img = Image.open(p).convert("RGB")
        if self.augment and random.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
        img = self.tf(img)
        targets = []
        lp = self.lbl_dir / (p.stem + ".txt")
        if lp.exists():
            with open(lp) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls, cx, cy, w, h = map(float, parts[:5])
                        if self.augment and random.random() < 0.5:  # already flipped image
                            cx = 1.0 - cx
                        targets.append([0, cls, cx, cy, w, h])
        t = torch.tensor(targets, dtype=torch.float32) if targets else torch.zeros((0, 6))
        return img, t

def collate(batch):
    imgs, tgts = zip(*batch)
    imgs = torch.stack(imgs)
    new = []
    for i, t in enumerate(tgts):
        if t.numel():
            t = t.clone(); t[:, 0] = i; new.append(t)
    return imgs, torch.cat(new, 0) if new else torch.zeros((0, 6))

train_ds = YoloDS(IMG_DIR, LBL_DIR, IMGSZ, max_n=None, augment=True)
val_ds   = YoloDS(IMG_DIR, LBL_DIR, IMGSZ, max_n=None, augment=False)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, collate_fn=collate, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, collate_fn=collate, num_workers=0)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## 4. Stronger YOLO Loss (DFL + multi-positive + CIoU) — Option C

In [ ]:
def bbox_ciou(box1, box2, eps=1e-7):
    # box1/2: (..., 4) xyxy
    b1x1, b1y1, b1x2, b1y2 = box1.unbind(-1)
    b2x1, b2y1, b2x2, b2y2 = box2.unbind(-1)
    inter = (torch.min(b1x2, b2x2) - torch.max(b1x1, b2x1)).clamp(0) * \
            (torch.min(b1y2, b2y2) - torch.max(b1y1, b2y1)).clamp(0)
    w1, h1 = b1x2 - b1x1, b1y2 - b1y1
    w2, h2 = b2x2 - b2x1, b2y2 - b2y1
    union = w1*h1 + w2*h2 - inter + eps
    iou = inter / union
    cw = torch.max(b1x2, b2x2) - torch.min(b1x1, b2x1)
    ch = torch.max(b1y2, b2y2) - torch.min(b1y1, b2y1)
    c2 = cw**2 + ch**2 + eps
    rho2 = ((b2x1 + b2x2 - b1x1 - b1x2)**2 + (b2y1 + b2y2 - b1y1 - b1y2)**2) / 4
    v = (4 / math.pi**2) * (torch.atan(w2/(h2+eps)) - torch.atan(w1/(h1+eps)))**2
    with torch.no_grad():
        alpha = v / (v - iou + 1 + eps)
    return iou - (rho2 / c2 + v * alpha)

class StrongYOLOLoss(nn.Module):
    """
    Stronger loss (Option C):
    - Multi-positive assignment: center cell + 3x3 neighborhood
    - DFL (Distribution Focal Loss) on the 4 sides
    - CIoU on decoded boxes
    - BCE on classification
    """
    def __init__(self, nc=9, reg_max=16, box_w=7.5, cls_w=0.5, dfl_w=1.5, radius=1):
        super().__init__()
        self.nc = nc
        self.reg_max = reg_max
        self.box_w = box_w
        self.cls_w = cls_w
        self.dfl_w = dfl_w
        self.radius = radius
        self.bce = nn.BCEWithLogitsLoss(reduction="none")
        self.register_buffer("project", torch.arange(reg_max, dtype=torch.float))

    def _dfl_loss(self, pred_dist, target_ltrb):
        # pred_dist: (N, 4, reg_max), target_ltrb: (N, 4) continuous
        target_ltrb = target_ltrb.clamp(0, self.reg_max - 1 - 0.01)
        tl = target_ltrb.long()
        tr = tl + 1
        wl = tr.float() - target_ltrb
        wr = 1 - wl
        loss = 0
        for k in range(4):
            loss = loss + (
                F.cross_entropy(pred_dist[:, k], tl[:, k], reduction="none") * wl[:, k] +
                F.cross_entropy(pred_dist[:, k], tr[:, k].clamp(max=self.reg_max-1), reduction="none") * wr[:, k]
            ).mean()
        return loss / 4

    def forward(self, preds, targets):
        device = preds[0].device
        if targets.numel() == 0:
            return sum(p.float().pow(2).mean() for p in preds) * 0.01

        total = torch.zeros(1, device=device)
        n_pos = 0

        for pred in preds:
            B, C, H, W = pred.shape
            stride = 640.0 / H
            pred = pred.permute(0, 2, 3, 1).contiguous()  # B,H,W,C
            box_raw = pred[..., : self.reg_max * 4].view(B, H, W, 4, self.reg_max)
            cls_raw = pred[..., self.reg_max * 4 :]       # B,H,W,nc

            for b in range(B):
                t = targets[targets[:, 0] == b]
                if t.numel() == 0:
                    # light bg push
                    total = total + self.bce(cls_raw[b], torch.zeros_like(cls_raw[b])).mean() * self.cls_w * 0.05
                    continue

                for gt in t:
                    cid = int(gt[1].item())
                    if not (0 <= cid < self.nc):
                        continue
                    cx, cy, bw, bh = gt[2:6]
                    gj0 = int(cy * H)
                    gi0 = int(cx * W)

                    # multi-positive: center + radius neighborhood
                    for dj in range(-self.radius, self.radius + 1):
                        for di in range(-self.radius, self.radius + 1):
                            gj = min(max(gj0 + dj, 0), H - 1)
                            gi = min(max(gi0 + di, 0), W - 1)

                            # classification
                            tcls = torch.zeros(self.nc, device=device)
                            tcls[cid] = 1.0
                            total = total + self.bce(cls_raw[b, gj, gi], tcls).mean() * self.cls_w

                            # DFL target: distance from grid cell center to gt sides (in grid units)
                            px = (gi + 0.5) / W
                            py = (gj + 0.5) / H
                            # gt xyxy normalized
                            gx1, gy1 = cx - bw/2, cy - bh/2
                            gx2, gy2 = cx + bw/2, cy + bh/2
                            # ltrb in units of stride / 640  → scale to reg bins
                            l = (px - gx1) * W
                            t_ = (py - gy1) * H
                            r = (gx2 - px) * W
                            b_ = (gy2 - py) * H
                            target_ltrb = torch.stack([l, t_, r, b_]).to(device).clamp(0, self.reg_max - 1 - 0.01)

                            # DFL loss
                            pred_dist = box_raw[b, gj, gi]  # 4, reg_max
                            total = total + self._dfl_loss(pred_dist.unsqueeze(0), target_ltrb.unsqueeze(0)) * self.dfl_w

                            # CIoU on decoded box
                            dist = (F.softmax(pred_dist, dim=1) * self.project.to(device)).sum(1)
                            pred_xyxy = torch.stack([
                                px - dist[0] * stride / 640,
                                py - dist[1] * stride / 640,
                                px + dist[2] * stride / 640,
                                py + dist[3] * stride / 640,
                            ])
                            gt_xyxy = torch.stack([gx1, gy1, gx2, gy2]).to(device)
                            iou = bbox_ciou(pred_xyxy.unsqueeze(0), gt_xyxy.unsqueeze(0))
                            total = total + (1.0 - iou).mean() * self.box_w
                            n_pos += 1

        return total / max(n_pos, 1) if n_pos else total + 1e-4

loss_fn = StrongYOLOLoss(nc=NC, reg_max=16, radius=1).to(DEVICE)
print("StrongYOLOLoss ready (DFL + multi-positive + CIoU)")

## 5. mAP@0.5 evaluator

In [ ]:
def decode_raw(preds, conf_thres=0.2, reg_max=16, nc=9, imgsz=640, topk=100):
    device = preds[0].device
    B = preds[0].shape[0]
    project = torch.arange(reg_max, device=device, dtype=torch.float)
    results = [[] for _ in range(B)]
    for pred in preds:
        B, C, H, W = pred.shape
        stride = imgsz / H
        pred = pred.permute(0, 2, 3, 1)
        box_raw = pred[..., :reg_max*4].view(B, H, W, 4, reg_max)
        cls_raw = pred[..., reg_max*4:]
        dist = (F.softmax(box_raw, -1) * project).sum(-1)
        gy, gx = torch.meshgrid(torch.arange(H, device=device), torch.arange(W, device=device), indexing="ij")
        cx = (gx + 0.5) * stride
        cy = (gy + 0.5) * stride
        boxes = torch.stack([cx - dist[...,0]*stride, cy - dist[...,1]*stride,
                             cx + dist[...,2]*stride, cy + dist[...,3]*stride], -1)
        scores, cls_ids = cls_raw.sigmoid().max(-1)
        mask = scores > conf_thres
        for b in range(B):
            m = mask[b]
            if m.any():
                det = torch.cat([boxes[b][m], scores[b][m, None], cls_ids[b][m, None].float()], 1)
                results[b].append(det)
    out = []
    for b in range(B):
        if results[b]:
            d = torch.cat(results[b], 0)
            if d.shape[0] > topk:
                d = d[d[:, 4].argsort(descending=True)[:topk]]
            out.append(d)
        else:
            out.append(torch.zeros((0, 6), device=device))
    return out

def box_iou_mat(a, b, eps=1e-7):
    aa = (a[:,2]-a[:,0]).clamp(0)*(a[:,3]-a[:,1]).clamp(0)
    bb = (b[:,2]-b[:,0]).clamp(0)*(b[:,3]-b[:,1]).clamp(0)
    lt = torch.max(a[:, None, :2], b[:, :2])
    rb = torch.min(a[:, None, 2:], b[:, 2:])
    wh = (rb - lt).clamp(0)
    inter = wh[..., 0] * wh[..., 1]
    return inter / (aa[:, None] + bb - inter + eps)

@torch.no_grad()
def evaluate_map50(model, loader, nc=9, conf=0.15, iou_thr=0.5):
    model.eval()
    all_preds = []
    all_gts = []  # list of (N,5) cls+xyxy abs
    for imgs, targets in loader:
        imgs = imgs.to(DEVICE)
        preds = model(imgs)
        decoded = decode_raw(preds, conf_thres=conf, nc=nc)
        all_preds.extend(decoded)
        for i in range(imgs.shape[0]):
            t = targets[targets[:, 0] == i]
            if t.numel():
                boxes = torch.stack([
                    (t[:,2]-t[:,4]/2)*IMGSZ, (t[:,3]-t[:,5]/2)*IMGSZ,
                    (t[:,2]+t[:,4]/2)*IMGSZ, (t[:,3]+t[:,5]/2)*IMGSZ], -1)
                all_gts.append(torch.cat([t[:, 1:2], boxes], 1))
            else:
                all_gts.append(torch.zeros((0, 5), device=DEVICE))

    aps = []
    for c in range(nc):
        preds_c, gts_c, n_gt = [], [], 0
        for p, g in zip(all_preds, all_gts):
            if p.numel():
                pc = p[p[:, 5] == c]
                if pc.numel(): preds_c.append(pc)
            gc = g[g[:, 0] == c]
            if gc.numel():
                gts_c.append(gc[:, 1:])
                n_gt += gc.shape[0]
        if n_gt == 0:
            continue
        if not preds_c:
            aps.append(0.0)
            continue
        pred = torch.cat(preds_c, 0)
        pred = pred[pred[:, 4].argsort(descending=True)]
        gt_all = torch.cat(gts_c, 0)
        matched = torch.zeros(gt_all.shape[0], dtype=torch.bool, device=DEVICE)
        tp = np.zeros(pred.shape[0])
        fp = np.zeros(pred.shape[0])
        for i in range(pred.shape[0]):
            if gt_all.numel() == 0:
                fp[i] = 1
                continue
            ious = box_iou_mat(pred[i, :4].unsqueeze(0), gt_all)[0]
            max_iou, j = ious.max(0)
            if max_iou >= iou_thr and not matched[j]:
                tp[i] = 1
                matched[j] = True
            else:
                fp[i] = 1
        tp_c = np.cumsum(tp)
        fp_c = np.cumsum(fp)
        rec = tp_c / (n_gt + 1e-8)
        prec = tp_c / (tp_c + fp_c + 1e-8)
        mrec = np.concatenate(([0.], rec, [1.]))
        mpre = np.concatenate(([1.], prec, [0.]))
        for i in range(mpre.size - 1, 0, -1):
            mpre[i-1] = np.maximum(mpre[i-1], mpre[i])
        idx = np.where(mrec[1:] != mrec[:-1])[0]
        ap = float(np.sum((mrec[idx+1] - mrec[idx]) * mpre[idx+1]))
        aps.append(ap)
    return {"mAP50": float(np.mean(aps)) if aps else 0.0, "AP_per_class": aps}

print("mAP evaluator ready")

## 6. Train (Option B clean re-train)

In [ ]:
model = build_mamba3yolo("T", nc=NC, is_mimo=False, d_state=64).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {n_params:,} | HAS_MAMBA3={HAS_MAMBA3}")

# optional: warm-start from previous best if you want
# ckpt = torch.load(RUN_DIR / "best.pt", map_location="cpu")
# model.load_state_dict(ckpt["model"], strict=False)

opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=5e-2)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)

SAVE = WORK / "runs" / "mamba3yolo" / f"real_{datetime.now().strftime('%m%d_%H%M')}"
SAVE.mkdir(parents=True, exist_ok=True)
print("Saving to", SAVE)

history = {"epoch": [], "loss": [], "mAP50": [], "lr": [], "time": []}
best_map = -1.0

print(f"🚀 Real training for {EPOCHS} epochs...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    total, n = 0.0, 0
    for imgs, targets in train_loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        preds = model(imgs)
        loss = loss_fn(preds, targets)
        if not torch.isfinite(loss):
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        opt.step()
        total += loss.item()
        n += 1
    sched.step()
    avg_loss = total / max(n, 1)
    dt = time.time() - t0
    lr = sched.get_last_lr()[0]

    # mAP every 10 epochs + last
    map50 = None
    if epoch % 10 == 0 or epoch == EPOCHS:
        metrics = evaluate_map50(model, val_loader, nc=NC, conf=0.15)
        map50 = metrics["mAP50"]
        print(f"Epoch {epoch:03d}/{EPOCHS} | loss={avg_loss:.4f} | mAP50={map50:.4f} | lr={lr:.2e} | {dt:.1f}s")
        if map50 > best_map:
            best_map = map50
            torch.save({"model": model.state_dict(), "epoch": epoch, "mAP50": map50,
                        "cfg": {"scale":"T","nc":NC,"is_mimo":False}, "has_mamba3": HAS_MAMBA3},
                       SAVE / "best.pt")
            print("  ★ new best mAP50")
    else:
        print(f"Epoch {epoch:03d}/{EPOCHS} | loss={avg_loss:.4f} | lr={lr:.2e} | {dt:.1f}s")

    history["epoch"].append(epoch)
    history["loss"].append(avg_loss)
    history["mAP50"].append(map50 if map50 is not None else (history["mAP50"][-1] if history["mAP50"] else 0.0))
    history["lr"].append(lr)
    history["time"].append(dt)

    torch.save({"model": model.state_dict(), "epoch": epoch, "history": history}, SAVE / "last.pt")

with open(SAVE / "history.json", "w") as f:
    json.dump(history, f, indent=2)
print(f"\n✅ Training done. Best mAP50 = {best_map:.4f}")
print("Artifacts →", SAVE)

## 7. Curves + summary

In [ ]:
sns.set_style("whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
axes[0].plot(history["epoch"], history["loss"], "b-")
axes[0].set_title("Train Loss"); axes[0].set_xlabel("Epoch")
# mAP only at logged points
map_epochs = [e for e, m in zip(history["epoch"], history["mAP50"]) if m is not None]
map_vals = [m for m in history["mAP50"] if m is not None]
if map_vals:
    axes[1].plot(map_epochs, map_vals, "g-o", ms=4)
axes[1].set_title("mAP@0.5"); axes[1].set_xlabel("Epoch"); axes[1].set_ylim(0, 1)
axes[2].plot(history["epoch"], history["lr"], "r-")
axes[2].set_title("LR"); axes[2].set_yscale("log")
plt.tight_layout()
fig.savefig(SAVE / "training_curves.png", dpi=200, bbox_inches="tight")
fig.savefig(SAVE / "training_curves.pdf", bbox_inches="tight")
plt.show()

summary = {
    "model": "Mamba3Yolo-T",
    "params": n_params,
    "HAS_MAMBA3": bool(HAS_MAMBA3),
    "dataset": "brain-tumor",
    "nc": NC,
    "epochs": EPOCHS,
    "best_mAP50": best_map,
    "final_loss": history["loss"][-1],
    "save_dir": str(SAVE),
    "timestamp": datetime.now().isoformat(),
}
with open(SAVE / "paper_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 55)
print("PAPER SUMMARY")
print("=" * 55)
for k, v in summary.items():
    print(f"  {k:20s}: {v}")
print("\nFiles:")
for p in sorted(SAVE.iterdir()):
    print(f"  {p.name:30s} {p.stat().st_size/1e6:.2f} MB")
print("\n✅ Done. Use best.pt + training_curves + paper_summary.json")

## ✅ Paper artifacts

| File | Use |
|------|-----|
| `best.pt` | Best mAP checkpoint |
| `training_curves.png/pdf` | Loss + mAP@0.5 curves |
| `paper_summary.json` | Numbers for tables |
| `history.json` | Full log |

**Tip:** For a quick test set `EPOCHS = 30`. For the paper use 100–150.